# Orders_Bronze_to_Silver

Reads the newest unprocessed order file from Bronze, cleanses and validates it, tags every row as valid or invalid, and stages the result. The pipeline's If-Condition then routes: valid rows to Silver on pass, the whole file to quarantine on fail.

**Achieves:** watermark-based new-file detection, schema-drift handling, row-level validation (missing fields, range checks, duplicates), staging in `orders_pending_routing`, one audit log row per run, raw-file archiving, and a quality-score payload for pipeline routing.

In [ ]:
from datetime import datetime
from pyspark.sql import Window
from pyspark.sql import functions as F
from notebookutils import mssparkutils
import json

bronze_dir = "Files/bronze/orders/"
archive_dir = bronze_dir + "archive/"
health_log_table_name = "silver_pipeline_audit"
QUALITY_THRESHOLD = 70.0

In [ ]:
# Incremental file discovery (watermark = SourceFile already logged in the
# audit table). Anything in Bronze not yet in the audit log is new; if
# nothing is new, exit early.
if spark.catalog.tableExists(health_log_table_name):
    processed_files = set(
        row.SourceFile for row in
        spark.sql(f"SELECT DISTINCT SourceFile FROM {health_log_table_name}").collect()
    )
else:
    processed_files = set()

all_files = mssparkutils.fs.ls(bronze_dir)
candidate_files = sorted(
    [f for f in all_files if f.name.endswith(".csv") and f.name not in processed_files],
    key=lambda x: x.name
)

if not candidate_files:
    # Idempotency: clear stale staged rows so a quiet hour with no new file
    # never lets an old file get reprocessed downstream.
    if spark.catalog.tableExists("orders_pending_routing"):
        empty_df = spark.table("orders_pending_routing").filter("1=0")
        empty_df.write.format("delta").mode("overwrite").saveAsTable("orders_pending_routing")
    mssparkutils.notebook.exit(json.dumps({"status": "no_new_files", "quality_score": 100.0}))

target_file = candidate_files[0]
bronze_file_path = target_file.path
bronze_file_name = target_file.name

df_bronze = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(bronze_file_path)
)

initial_count = df_bronze.count()

In [ ]:
# Cleansing and row-level validation. Legacy column names are mapped back
# to standard; every row is tagged IsValid (+ FailureReason) rather than
# dropped. Checks: missing mandatory fields, out-of-range values, duplicate
# OrderId (keep latest by LastModifiedDate).
if "OrderTotal" in df_bronze.columns and "TotalAmount" not in df_bronze.columns:
    df_bronze = df_bronze.withColumnRenamed("OrderTotal", "TotalAmount")
    print("-> Schema drift handled: 'OrderTotal' renamed to 'TotalAmount'")

df_formatted = (
    df_bronze.withColumn("CustomerId", F.trim(F.col("CustomerId")))
    .withColumn("OrderDate", F.to_timestamp("OrderDate"))
    .withColumn("LastModifiedDate", F.to_timestamp("LastModifiedDate"))
)

missing_mask = F.col("OrderId").isNull() | F.col("CustomerId").isNull() | F.col("Quantity").isNull()
range_ok = (F.col("Quantity") > 0) & (F.col("TotalAmount") >= 0)

window_latest = Window.partitionBy("OrderId").orderBy(F.col("LastModifiedDate").desc())
df_ranked = df_formatted.withColumn("row_num", F.row_number().over(window_latest))
is_latest = F.col("row_num") == 1

df_tagged = (
    df_ranked
    .withColumn(
        "FailureReason",
        F.when(missing_mask, "missing_mandatory_field")
         .when(~range_ok, "range_check_failed")
         .when(~is_latest, "duplicate_order_id")
         .otherwise(F.lit(None))
    )
    .withColumn("IsValid", (~missing_mask) & range_ok & is_latest)
    .drop("row_num")
)

nulls_removed = df_tagged.filter(F.col("FailureReason") == "missing_mandatory_field").count()
outliers_removed = df_tagged.filter(F.col("FailureReason") == "range_check_failed").count()
duplicates_removed = df_tagged.filter(F.col("FailureReason") == "duplicate_order_id").count()
final_count = df_tagged.filter(F.col("IsValid")).count()
quality_score = round((final_count / initial_count) * 100, 2) if initial_count > 0 else 0.0

print("=" * 60)
print(f" DATA QUALITY REPORT — {bronze_file_name}")
print("=" * 60)
print(f"-> Initial rows      : {initial_count}")
print(f"-> Nulls removed     : {nulls_removed}")
print(f"-> Outliers removed  : {outliers_removed}")
print(f"-> Duplicates removed: {duplicates_removed}")
print(f"-> Valid rows        : {final_count}")
print(f"-> Quality score     : {quality_score}%")
print("=" * 60)

In [ ]:
# Stage the tagged dataset and log the audit entry. orders_pending_routing
# is overwritten each run — always exactly the current file's tagged rows.
# Both downstream notebooks (Silver merge, Quarantine) read from this one
# table. One audit row is appended per run regardless of outcome.
df_tagged.write.format("delta").mode("overwrite").saveAsTable("orders_pending_routing")

health_row = [{
    "RunTimestamp": datetime.now(),
    "SourceFile": bronze_file_name,
    "InitialRows": int(initial_count),
    "NullsRemoved": int(nulls_removed),
    "OutliersRemoved": int(outliers_removed),
    "DuplicatesRemoved": int(duplicates_removed),
    "CleanRows": int(final_count),
    "QualityScore": float(quality_score),
    "PassedThreshold": bool(quality_score >= QUALITY_THRESHOLD),
}]

spark.createDataFrame(health_row).write.format("delta").mode("append").saveAsTable(
    health_log_table_name
)

In [ ]:
# Archive the raw file (always, regardless of outcome) and hand the
# quality score back to the pipeline, which decides the next step.
mssparkutils.fs.mkdirs(archive_dir)
mssparkutils.fs.mv(bronze_file_path, archive_dir + bronze_file_name, overwrite=True)
print(f"-> {bronze_file_name} moved to archive.")

status = "passed_quality_gate" if quality_score >= QUALITY_THRESHOLD else "failed_quality_gate"

payload = {
    "status": status,
    "quality_score": float(quality_score),
    "file": bronze_file_name,
}

mssparkutils.notebook.exit(json.dumps(payload))